In [6]:
# TODO: Import necessary libraries
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
# TODO: Set pandas display options
# - display.max_columns to None
# - display.max_rows to 100
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# TODO: Print success message
print("Environment setup complete.")

Environment setup complete.


In [7]:
# TODO: Load cleaned data from Phase 1
# - Read 'train_cleaned.csv' using pd.read_csv()
# - Store in variable 'df'
df = pd.read_csv('../data/satilir_properties_cleaned.csv')
# TODO: Print dataset information
# - Dataset shape
# - Total missing values
# - Display first few rows with .head()
print("Dataset Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())
df.head()

Dataset Shape: (4416, 25)
Total Missing Values: 0


,price,rooms,area_m2,land_area_sot,floor,has_document,address,avtodayanacaq,balkon,duzelme,esyali,hovuz,internet,isiq,kabel_tv,kombi,kondisioner,lift,merkezi_qizdirici_sistem,metbex_mebeli,pvc_pencere,qaz,su,telefon,temirsiz
0,200000,3.0,65.0,0.0,5.0,No,"Bakı, Nərimanov, metro 28 May",Yes,Yes,No,No,No,Yes,Yes,Yes,Yes,Yes,Yes,No,Yes,Yes,Yes,Yes,No,No
1,210000,2.0,54.0,0.0,12.0,No,"Bakı, Nərimanov, metro Nəriman Nərimanov",No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No
2,149000,4.0,100.0,0.0,2.0,Yes,"Bakı, Binəqədi, Biləcəri qəs.",No,No,No,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No
3,45000,1.0,44.0,0.0,1.0,Yes,Xırdalan,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No
4,109000,3.0,69.0,0.0,13.0,No,Xırdalan,Yes,Yes,No,No,No,Yes,Yes,Yes,Yes,No,Yes,No,No,Yes,Yes,Yes,No,No


In [8]:
# Detect Yes/No columns (encoding will be handled inside the preprocessing pipeline later)
yes_no_cols = []
for col in df.columns:
    non_null = df[col].dropna().astype(str).str.strip().str.lower()
    if len(non_null) and set(non_null.unique()).issubset({"yes", "no"}):
        yes_no_cols.append(col)

print("Detected Yes/No columns:", yes_no_cols)

Detected Yes/No columns: ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirsiz']


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4416 entries, 0 to 4415
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   price                     4416 non-null   int64  
 1   rooms                     4416 non-null   float64
 2   area_m2                   4416 non-null   float64
 3   land_area_sot             4416 non-null   float64
 4   floor                     4416 non-null   float64
 5   has_document              4416 non-null   object 
 6   address                   4416 non-null   object 
 7   avtodayanacaq             4416 non-null   object 
 8   balkon                    4416 non-null   object 
 9   duzelme                   4416 non-null   object 
 10  esyali                    4416 non-null   object 
 11  hovuz                     4416 non-null   object 
 12  internet                  4416 non-null   object 
 13  isiq                      4416 non-null   object 
 14  kabel_tv

In [10]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print("Count of Numerical Columns:", len(num_cols))
print("Numerical Columns:", num_cols)

Count of Numerical Columns: 5
Numerical Columns: ['price', 'rooms', 'area_m2', 'land_area_sot', 'floor']


In [11]:
df["address"].unique()

array(['Bakı, Nərimanov, metro 28 May',
       'Bakı, Nərimanov, metro Nəriman Nərimanov',
       'Bakı, Binəqədi, Biləcəri qəs.', 'Xırdalan', 'Bakı, Nəsimi',
       'Bakı, Nizami, metro Qara Qarayev',
       'Bakı, Xətai, Əhmədli, metro Həzi Aslanov',
       'Bakı, Suraxanı, Yeni Günəşli qəs.',
       'Bakı, Yasamal, metro İnşaatçılar', 'Bakı, Abşeron, Masazır',
       'Bakı, Binəqədi, Biləcəri qəs., metro Avtovağzal',
       'Bakı, Xəzər, Buzovna',
       'Bakı, Nizami, 8-ci kilometr, metro Neftçilər',
       'Bakı, Xətai, Əhmədli, metro Əhmədli',
       'Bakı, Yasamal, Yeni Yasamal qəs., metro İnşaatçılar',
       'Bakı, Abşeron, Qobu',
       'Bakı, Yasamal, Yasamal qəs., metro İnşaatçılar',
       'Bakı, Binəqədi, metro Azadlıq',
       'Bakı, Yasamal, Yeni Yasamal qəs., metro İçərişəhər',
       'Bakı, Binəqədi, 9-cu mikrorayon', 'Bakı, Sabunçu, metro Koroğlu',
       'Bakı, Nizami, metro Neftçilər', 'Bakı, Yasamal, metro 20 Yanvar',
       'Bakı, Xətai, H.Aslanov qəs.',
       '

In [36]:
import re


def _normalize_loc_token(x):
    token = '' if pd.isna(x) else str(x)
    token = token.strip().lower()
    token = token.replace('qəs.', 'qes').replace('qəs', 'qes')
    if token == '' or token == 'nan':
        return 'unknown'
    return token


def extract_address_parts(address_series: pd.Series) -> pd.DataFrame:
    out = pd.DataFrame(index=address_series.index)

    city_vals = []
    district_vals = []
    neighborhood_vals = []
    metro_vals = []
    metro_present_vals = []

    for raw in address_series.fillna('').astype(str):
        txt = raw.strip()
        parts = [p.strip() for p in txt.split(',') if p.strip()]
        parts_norm = [_normalize_loc_token(p) for p in parts]

        city = parts_norm[0] if len(parts_norm) >= 1 else 'unknown'

        district = 'unknown'
        for p in parts_norm[1:]:
            if 'metro' not in p:
                district = p
                break

        neighborhood = 'unknown'
        if len(parts_norm) >= 3:
            for p in parts_norm[2:]:
                if 'metro' not in p:
                    neighborhood = p
                    break

        metro_match = re.search(r'metro\s+([^,]+)', txt, flags=re.IGNORECASE)
        metro = _normalize_loc_token(metro_match.group(1)) if metro_match else 'unknown'
        metro_present = 0 if metro == 'unknown' else 1

        city_vals.append(city)
        district_vals.append(district)
        neighborhood_vals.append(neighborhood)
        metro_vals.append(metro)
        metro_present_vals.append(metro_present)

    out['loc_city_raw'] = city_vals
    out['loc_district_raw'] = district_vals
    out['loc_neighborhood_raw'] = neighborhood_vals
    out['loc_metro_raw'] = metro_vals
    out['num__metro_present_v6'] = metro_present_vals
    return out

In [40]:
if 'address' in df.columns:
    address_series = df['address']
else:
    address_series = pd.read_csv('../data/satilir_properties_cleaned.csv', usecols=['address'])['address']
    if len(address_series) != len(df):
        raise ValueError(f'Row mismatch while recovering address: address={len(address_series)}, df={len(df)}')
    address_series = pd.Series(address_series.values, index=df.index)

location_parts = extract_address_parts(address_series)

address_parts = pd.DataFrame(
    {
        'address_part_1': location_parts['loc_city_raw'].replace('unknown', pd.NA),
        'address_part_2': location_parts['loc_district_raw'].replace('unknown', pd.NA),
        'address_part_3': location_parts['loc_neighborhood_raw'].replace('unknown', pd.NA),
        'address_part_4': location_parts['loc_metro_raw'].replace('unknown', pd.NA),
    },
    index=df.index,
 )

for col in address_parts.columns:
    df[col] = address_parts[col]

print('Created address part columns:', address_parts.columns.tolist())
df[address_parts.columns].head()

Created address part columns: ['address_part_1', 'address_part_2', 'address_part_3', 'address_part_4']


,address_part_1,address_part_2,address_part_3,address_part_4
0,bakı,nərimanov,<NA>,28 may
1,bakı,nərimanov,<NA>,nəriman nərimanov
2,bakı,binəqədi,biləcəri qes,<NA>
3,xırdalan,<NA>,<NA>,<NA>
4,xırdalan,<NA>,<NA>,<NA>


In [13]:
df.drop(columns=["address"], inplace=True)

In [14]:
df[address_parts.columns].isnull().sum()

address_part_1       0
address_part_2     632
address_part_3     959
address_part_4    3372
dtype: int64

In [15]:
df[address_parts.columns].value_counts(dropna=False)

address_part_1  address_part_2  address_part_3           address_part_4       
Bakı            Abşeron         Masazır                  NaN                      632
Xırdalan        NaN             NaN                      NaN                      606
Bakı            Yasamal         Yeni Yasamal qəs.        metro İnşaatçılar        192
                Suraxanı        Yeni Günəşli qəs.        NaN                      162
                Nərimanov       metro Nəriman Nərimanov  NaN                      154
                                                                                 ... 
                Binəqədi        9-cu mikrorayon          metro Memar Əcəmi - 2      1
                Yasamal         Yeni Yasamal qəs.        metro İçərişəhər           1
                Nəsimi          3-cü mikrorayon          metro 20 Yanvar            1
                Xətai           H.Aslanov qəs.           metro Xətai                1
                Abşeron         Ceyranbatan qəs.         NaN 

In [16]:
df['address_part_1'].unique()

array(['Bakı', 'Xırdalan', 'Lənkəran', 'Sumqayıt', 'Qəbələ', 'Siyazən'],
      dtype=object)

In [17]:
df['address_part_2'].unique()

array(['Nərimanov', 'Binəqədi', None, 'Nəsimi', 'Nizami', 'Xətai',
       'Suraxanı', 'Yasamal', 'Abşeron', 'Xəzər', 'Sabunçu', 'Səbail',
       '10-cu mikrorayon', '1-ci mikrorayon', 'St.Sumqayıt',
       '18-ci mikrorayon', '44-cü məhəllə', 'Qaradağ', 'Sumqayıt Bulvarı',
       '14-cü məhəllə', '9-cu mikrorayon', '6-ci mikrorayon',
       '72-ci məhəllə', '17-ci mikrorayon', '16-ci mikrorayon',
       '8-ci mikrorayon', '12-ci mikrorayon', '13-cü mikrorayon',
       '5-ci mikrorayon', '1-ci məhəllə', 'metro Memar Əcəmi',
       '36-ci məhəllə', '2-ci mikrorayon', '3-ci məhəllə',
       '15-ci məhəllə', '40-ci məhəllə'], dtype=object)

In [18]:
print("Before deleting rows, dataset shape:", df.shape)
df = df[~((df["address_part_1"] == "Bakı") & (df["address_part_2"].isnull()))]
print("After deleting rows, dataset shape:", df.shape)

Before deleting rows, dataset shape: (4416, 28)
After deleting rows, dataset shape: (4416, 28)


In [19]:
df[(df["address_part_1"] != "Bakı") & (df["address_part_2"].isnull())]

,price,rooms,area_m2,land_area_sot,floor,has_document,avtodayanacaq,balkon,duzelme,esyali,hovuz,internet,isiq,kabel_tv,kombi,kondisioner,lift,merkezi_qizdirici_sistem,metbex_mebeli,pvc_pencere,qaz,su,telefon,temirsiz,address_part_1,address_part_2,address_part_3,address_part_4
3,45000,1.0,44.0,0.0,1.0,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Xırdalan,None,None,None
4,109000,3.0,69.0,0.0,13.0,No,Yes,Yes,No,No,No,Yes,Yes,Yes,Yes,No,Yes,No,No,Yes,Yes,Yes,No,No,Xırdalan,None,None,None
5,70500,2.0,73.0,0.0,4.0,No,Yes,No,No,No,No,No,Yes,Yes,Yes,No,No,No,No,Yes,Yes,Yes,No,No,Xırdalan,None,None,None
7,135000,3.0,100.0,0.0,9.0,No,Yes,Yes,No,Yes,No,Yes,Yes,Yes,Yes,No,Yes,No,Yes,Yes,Yes,Yes,No,No,Xırdalan,None,None,None
8,54000,1.0,31.0,0.0,5.0,No,Yes,No,No,No,No,Yes,Yes,Yes,Yes,No,Yes,No,No,Yes,Yes,Yes,No,No,Xırdalan,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4397,92000,2.0,65.0,0.0,2.0,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Xırdalan,None,None,None
4406,92000,3.0,66.0,0.0,1.0,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Xırdalan,None,None,None
4407,120000,4.0,96.0,0.0,1.0,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Xırdalan,None,None,None
4408,230000,4.0,164.0,0.0,11.0,Yes,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Xırdalan,None,None,None


In [20]:
# df[(df["cities"] != "Bakı") & (df["address_part_2"].isnull()), "address_part_2"]
df.loc[(df["address_part_2"] != "Bakı") & (df["address_part_2"].isnull()), "address_part_2"] = df.loc[(df["address_part_1"] != "Bakı") & (df["address_part_2"].isnull()), "address_part_1"]

In [21]:
df.loc[(df["address_part_1"] != "Bakı"), ["address_part_1" , "address_part_2"]]

,address_part_1,address_part_2
3,Xırdalan,Xırdalan
4,Xırdalan,Xırdalan
5,Xırdalan,Xırdalan
7,Xırdalan,Xırdalan
8,Xırdalan,Xırdalan
...,...,...
4397,Xırdalan,Xırdalan
4406,Xırdalan,Xırdalan
4407,Xırdalan,Xırdalan
4408,Xırdalan,Xırdalan


In [22]:
df[address_parts.columns].isnull().sum()

address_part_1       0
address_part_2       0
address_part_3     959
address_part_4    3372
dtype: int64

In [23]:
df['address_part_3'].unique()

array(['metro 28 May', 'metro Nəriman Nərimanov', 'Biləcəri qəs.', None,
       'metro Qara Qarayev', 'Əhmədli', 'Yeni Günəşli qəs.',
       'metro İnşaatçılar', 'Masazır', 'Buzovna', '8-ci kilometr',
       'Yeni Yasamal qəs.', 'Qobu', 'Yasamal qəs.', 'metro Azadlıq',
       '9-cu mikrorayon', 'metro Koroğlu', 'metro Neftçilər',
       'metro 20 Yanvar', 'H.Aslanov qəs.', 'metro Memar Əcəmi',
       'metro İçərişəhər', '8-ci mikrorayon', 'metro Həzi Aslanov',
       'metro Xalqlar dostluğu', 'metro Xətai',
       'metro Elmlər akademiyası', '7-ci mikrorayon', 'metro 8 Noyabr',
       'Saray', 'metro Gənclik', 'Bakıxanov qəs.', 'metro Nizami',
       'Badamdar qəs.', 'Qara şəhər', 'metro Əhmədli', 'metro Sahil',
       'Nardaran qəs.', '3-cü mikrorayon', 'Köhnə Günəşli qəs.',
       'metro Nəsimi', 'Əmircan qəs.', 'Lökbatan qəs.', 'Qaraçuxur qəs.',
       'Hövsan qəs.', 'Ramana qəs.', 'Ağ şəhər', '6-cı mikrorayon',
       'Bayıl qəs.', 'Binəqədi qəs.', 'Montin qəs.', 'Zığ qəs.',
      

In [24]:
df['address_part_4'].unique()

array([None, 'metro Həzi Aslanov', 'metro Avtovağzal', 'metro Neftçilər',
       'metro Əhmədli', 'metro İnşaatçılar', 'metro İçərişəhər',
       'metro Nəsimi', 'metro Azadlıq', 'metro Elmlər akademiyası',
       'metro 20 Yanvar', 'metro Dərnəgül', 'metro Xalqlar dostluğu',
       'metro Gənclik', 'metro Qara Qarayev', 'metro Koroğlu',
       'metro Xətai', 'metro Memar Əcəmi', 'metro Nəriman Nərimanov',
       'metro Memar Əcəmi - 2', 'metro 8 Noyabr', 'metro 28 May',
       'metro Nizami', 'metro Xocaəsən', 'metro Sahil'], dtype=object)

In [25]:
df.drop(columns=["address_part_3", "address_part_4"], inplace=True)

In [26]:
df.shape

(4416, 26)

In [27]:
df.rename(columns={"address_part_1": "cities"}, inplace=True)
df.rename(columns={"address_part_2": "raion"}, inplace=True)

In [28]:
print("Number of unique cities:", df['cities'].nunique())
print(f"Cardinality of cities: {(df['cities'].nunique()/df.shape[0]) * 100:.2f}%")

Number of unique cities: 6
Cardinality of cities: 0.14%


In [29]:
print("Number of unique cities:", df['raion'].nunique())
print(f"Cardinality of cities: {(df['raion'].nunique()/df.shape[0]) * 100:.2f}%")

Number of unique cities: 40
Cardinality of cities: 0.91%


In [30]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

class YesNoBinaryEncoder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.feature_names_in_ = X_df.columns.to_numpy()
        return self

    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.feature_names_in_)
        out = pd.DataFrame(index=X_df.index)
        for col in self.feature_names_in_:
            out[col] = (
                X_df[col]
                .astype("string")
                .str.strip()
                .str.lower()
                .map({"yes": 1, "no": 0})
                .astype(float)
            )
        return out.to_numpy()

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        return np.asarray(input_features, dtype=object)

class MultiColumnFrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.freq_maps_ = {}
        self.columns_ = []

    def fit(self, X, y=None):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        self.columns_ = X_df.columns.tolist()
        self.feature_names_in_ = np.asarray(self.columns_, dtype=object)
        self.freq_maps_ = {
            col: X_df[col].astype("string").value_counts(normalize=True, dropna=False).to_dict()
            for col in self.columns_
        }
        return self

    def transform(self, X):
        X_df = X.copy() if isinstance(X, pd.DataFrame) else pd.DataFrame(X, columns=self.columns_)
        out = pd.DataFrame(index=X_df.index)
        for col in self.columns_:
            out[f"{col}_freq"] = (
                X_df[col].astype("string").map(self.freq_maps_[col]).fillna(0.0).astype(float)
            )
        return out.to_numpy()

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            input_features = self.feature_names_in_
        input_features = np.asarray(input_features, dtype=object)
        return np.asarray([f"{col}_freq" for col in input_features], dtype=object)

categorical_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
categorical_cols = [c for c in categorical_cols if c not in yes_no_cols]

high_card_cols = [c for c in categorical_cols if df[c].nunique(dropna=False) > 20]
low_card_cols = [c for c in categorical_cols if c not in high_card_cols]

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

print("Yes/No columns:", yes_no_cols)
print("Low-cardinality categorical columns:", low_card_cols)
print("High-cardinality categorical columns:", high_card_cols)
print("Numeric columns:", len(numeric_cols))

preprocessor = ColumnTransformer(
    transformers=[
        (
            "yes_no_binary",
            Pipeline([("yes_no", YesNoBinaryEncoder())]),
            yes_no_cols
        ),
        (
            "cat_ohe",
            Pipeline([("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]),
            low_card_cols
        ),
        (
            "cat_freq",
            Pipeline([("freq", MultiColumnFrequencyEncoder())]),
            high_card_cols
        ),
        ("num", "passthrough", numeric_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=True,
    sparse_threshold=0
)

full_pipeline = Pipeline([("preprocessor", preprocessor)])

X_processed = full_pipeline.fit_transform(df)
feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()
processed_df = pd.DataFrame(X_processed, columns=feature_names, index=df.index)

print("Processed feature matrix shape:", processed_df.shape)
processed_df.head()

Yes/No columns: ['has_document', 'avtodayanacaq', 'balkon', 'duzelme', 'esyali', 'hovuz', 'internet', 'isiq', 'kabel_tv', 'kombi', 'kondisioner', 'lift', 'merkezi_qizdirici_sistem', 'metbex_mebeli', 'pvc_pencere', 'qaz', 'su', 'telefon', 'temirsiz']
Low-cardinality categorical columns: ['cities']
High-cardinality categorical columns: ['raion']
Numeric columns: 5
Processed feature matrix shape: (4416, 31)


,yes_no_binary__has_document,yes_no_binary__avtodayanacaq,yes_no_binary__balkon,yes_no_binary__duzelme,yes_no_binary__esyali,yes_no_binary__hovuz,yes_no_binary__internet,yes_no_binary__isiq,yes_no_binary__kabel_tv,yes_no_binary__kombi,yes_no_binary__kondisioner,yes_no_binary__lift,yes_no_binary__merkezi_qizdirici_sistem,yes_no_binary__metbex_mebeli,yes_no_binary__pvc_pencere,yes_no_binary__qaz,yes_no_binary__su,yes_no_binary__telefon,yes_no_binary__temirsiz,cat_ohe__cities_Bakı,cat_ohe__cities_Lənkəran,cat_ohe__cities_Qəbələ,cat_ohe__cities_Siyazən,cat_ohe__cities_Sumqayıt,cat_ohe__cities_Xırdalan,cat_freq__raion_freq,num__price,num__rooms,num__area_m2,num__land_area_sot,num__floor
0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.076087,200000.0,3.0,65.0,0.0,5.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.076087,210000.0,2.0,54.0,0.0,12.0
2,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.066350,149000.0,4.0,100.0,0.0,2.0
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.137228,45000.0,1.0,44.0,0.0,1.0
4,0.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.137228,109000.0,3.0,69.0,0.0,13.0


In [31]:
# TODO: Save the cleaned dataset to CSV
processed_df.to_csv('../data/satilir_properties_preprocessed.csv', index=False)
# - Print confirmation message
print("Preprocessed dataset saved to 'data/satilir_properties_preprocessed.csv'")

Preprocessed dataset saved to 'data/satilir_properties_preprocessed.csv'


## Model Feature Engineering Export (For Model Notebook)

This section prepares model-only inputs and saves them as files consumed by the modeling notebook.

Exports:
- `../data/satilir_model_base_v3.csv` (engineered model base features)
- `../data/satilir_location_parts.csv` (raw parsed location parts)
- `../data/satilir_location_encoded_v6.csv` (full-fit location encoding, reference)
- `../data/satilir_location_encoded_oof_v6.csv` (leakage-safer OOF location encoding for modeling)

In [41]:
from sklearn.model_selection import KFold


def _safe_qcut(series: pd.Series, q: int) -> pd.Series:
    try:
        ranked = series.rank(method='first')
        return pd.qcut(ranked, q=q, labels=False, duplicates='drop').astype(float)
    except Exception:
        return pd.Series(np.zeros(len(series), dtype=float), index=series.index)


def add_feature_block_v3(frame: pd.DataFrame) -> pd.DataFrame:
    xf = frame.copy()
    eps = 1e-6

    if 'num__area_m2' in xf.columns:
        area = xf['num__area_m2'].clip(lower=0)
        xf['num__log_area_m2_v3'] = np.log1p(area)
        xf['cat_bin__area_q5_v3'] = _safe_qcut(area, 5)

    if 'num__land_area_sot' in xf.columns:
        land = xf['num__land_area_sot'].clip(lower=0)
        xf['num__log_land_area_sot_v3'] = np.log1p(land)

    if 'num__floor' in xf.columns:
        floor = xf['num__floor'].clip(lower=0)
        xf['num__log_floor_v3'] = np.log1p(floor)

    if 'num__rooms' in xf.columns:
        rooms = xf['num__rooms'].clip(lower=0)
        xf['cat_bin__rooms_q4_v3'] = _safe_qcut(rooms, 4)

    if 'num__area_m2' in xf.columns and 'num__rooms' in xf.columns:
        xf['num__area_per_room_v3'] = xf['num__area_m2'] / (xf['num__rooms'] + eps)
        xf['num__rooms_per_area_v3'] = xf['num__rooms'] / (xf['num__area_m2'] + eps)
        xf['num__area_x_rooms_v3'] = xf['num__area_m2'] * xf['num__rooms']

    if 'num__area_m2' in xf.columns and 'num__floor' in xf.columns:
        xf['num__area_x_floor_v3'] = xf['num__area_m2'] * xf['num__floor']
        xf['num__floor_over_log_area_v3'] = xf['num__floor'] / (np.log1p(xf['num__area_m2'].clip(lower=0)) + eps)

    if 'num__land_area_sot' in xf.columns and 'num__area_m2' in xf.columns:
        xf['num__land_to_area_ratio_v3'] = xf['num__land_area_sot'] / (xf['num__area_m2'] + eps)
        xf['num__land_x_area_v3'] = xf['num__land_area_sot'] * xf['num__area_m2']

    if 'cat_bin__area_q5_v3' in xf.columns and 'cat_bin__rooms_q4_v3' in xf.columns:
        xf['num__area_bin_x_rooms_bin_v3'] = xf['cat_bin__area_q5_v3'] * 10 + xf['cat_bin__rooms_q4_v3']

    amenity_cols = [c for c in xf.columns if c.startswith('yes_no_binary__')]
    comfort_candidates = [
        'yes_no_binary__lift',
        'yes_no_binary__kondisioner',
        'yes_no_binary__kombi',
        'yes_no_binary__metbex_mebeli',
        'yes_no_binary__balkon',
        'yes_no_binary__pvc_pencere',
        'yes_no_binary__avtodayanacaq',
        'yes_no_binary__hovuz',
    ]
    utility_candidates = [
        'yes_no_binary__qaz',
        'yes_no_binary__su',
        'yes_no_binary__isiq',
        'yes_no_binary__internet',
        'yes_no_binary__kabel_tv',
        'yes_no_binary__telefon',
    ]

    comfort_cols = [c for c in comfort_candidates if c in xf.columns]
    utility_cols = [c for c in utility_candidates if c in xf.columns]

    if amenity_cols:
        xf['num__amenity_count_v3'] = xf[amenity_cols].sum(axis=1)
    if comfort_cols:
        xf['num__comfort_score_v3'] = xf[comfort_cols].sum(axis=1)
    if utility_cols:
        xf['num__utility_score_v3'] = xf[utility_cols].sum(axis=1)

    if 'num__comfort_score_v3' in xf.columns and 'num__rooms' in xf.columns:
        xf['num__comfort_x_rooms_v3'] = xf['num__comfort_score_v3'] * xf['num__rooms']

    if 'num__utility_score_v3' in xf.columns and 'num__area_m2' in xf.columns:
        xf['num__utility_per_area_v3'] = xf['num__utility_score_v3'] / (xf['num__area_m2'] + eps)

    if 'cat_freq__raion_freq' in xf.columns:
        xf['num__log_raion_freq_v3'] = np.log1p(xf['cat_freq__raion_freq'].clip(lower=0))
        if 'num__log_area_m2_v3' in xf.columns:
            xf['num__raionfreq_x_logarea_v3'] = xf['cat_freq__raion_freq'] * xf['num__log_area_m2_v3']

    return xf


def winsorize_series_v3(s: pd.Series, low_q: float = 0.01, high_q: float = 0.99) -> pd.Series:
    low_v, high_v = s.quantile([low_q, high_q])
    return s.clip(low_v, high_v)


def fit_location_maps(loc_train_df: pd.DataFrame, y_train: pd.Series, alpha: float = 35.0) -> dict:
    global_mean = float(y_train.mean())
    maps = {}

    for col in ['loc_city_raw', 'loc_district_raw', 'loc_neighborhood_raw', 'loc_metro_raw']:
        tmp = pd.DataFrame({'k': loc_train_df[col].fillna('unknown'), 'y': y_train.values})
        grp = tmp.groupby('k')['y'].agg(['mean', 'count'])
        smooth = (grp['mean'] * grp['count'] + global_mean * alpha) / (grp['count'] + alpha)
        maps[col] = smooth.to_dict()

    return {'global_mean': global_mean, 'maps': maps}


def apply_location_maps(loc_df: pd.DataFrame, fitted: dict) -> pd.DataFrame:
    xloc = pd.DataFrame(index=loc_df.index)

    xloc['num__city_te_v6'] = loc_df['loc_city_raw'].map(fitted['maps']['loc_city_raw']).fillna(fitted['global_mean'])
    xloc['num__district_te_v6'] = loc_df['loc_district_raw'].map(fitted['maps']['loc_district_raw']).fillna(fitted['global_mean'])
    xloc['num__neighborhood_te_v6'] = loc_df['loc_neighborhood_raw'].map(fitted['maps']['loc_neighborhood_raw']).fillna(fitted['global_mean'])
    xloc['num__metro_te_v6'] = loc_df['loc_metro_raw'].map(fitted['maps']['loc_metro_raw']).fillna(fitted['global_mean'])

    xloc['num__metro_present_v6'] = loc_df['num__metro_present_v6'].astype(float)
    xloc['num__location_te_mean_v6'] = xloc[['num__city_te_v6', 'num__district_te_v6', 'num__neighborhood_te_v6', 'num__metro_te_v6']].mean(axis=1)
    xloc['num__district_to_city_ratio_v6'] = xloc['num__district_te_v6'] / (xloc['num__city_te_v6'] + 1e-6)
    xloc['num__metro_to_district_ratio_v6'] = xloc['num__metro_te_v6'] / (xloc['num__district_te_v6'] + 1e-6)

    return xloc


def build_oof_location_encoded(
    loc_df: pd.DataFrame,
    y: pd.Series,
    n_splits: int = 5,
    alpha: float = 35.0,
    random_state: int = 42,
 ) -> pd.DataFrame:
    y_series = y if isinstance(y, pd.Series) else pd.Series(y, index=loc_df.index)
    y_series = y_series.loc[loc_df.index]

    cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_frames = []

    for tr_idx, va_idx in cv.split(loc_df):
        loc_tr = loc_df.iloc[tr_idx]
        loc_va = loc_df.iloc[va_idx]
        y_tr = y_series.iloc[tr_idx]

        fitted = fit_location_maps(loc_tr, y_tr, alpha=alpha)
        fold_frames.append(apply_location_maps(loc_va, fitted))

    oof_encoded = pd.concat(fold_frames).sort_index()
    oof_encoded = oof_encoded.loc[loc_df.index]

    if oof_encoded.isna().any().any():
        fallback = apply_location_maps(
            loc_df,
            fit_location_maps(loc_df, y_series, alpha=alpha),
        )
        oof_encoded = oof_encoded.combine_first(fallback)

    return oof_encoded

In [42]:
# Build and save model-feature inputs used by arenda-az-Model-Improved.ipynb
if 'processed_df' not in globals():
    processed_df = pd.read_csv('../data/satilir_properties_preprocessed.csv')

model_base_df = add_feature_block_v3(processed_df.copy())

addr_df = pd.read_csv('../data/satilir_properties_cleaned.csv', usecols=['address'])
location_parts_df = extract_address_parts(addr_df['address'])

if len(model_base_df) != len(location_parts_df):
    raise ValueError(f'Row mismatch: model_base_df={len(model_base_df)}, location_parts_df={len(location_parts_df)}')

if 'num__price' not in model_base_df.columns:
    raise KeyError("'num__price' column is required in model_base_df for location encoding.")

y_loc_full = winsorize_series_v3(model_base_df['num__price'], 0.01, 0.99)

# Full-fit map (reference export)
loc_fitted_full = fit_location_maps(location_parts_df, y_loc_full, alpha=35.0)
location_encoded_df = apply_location_maps(location_parts_df, loc_fitted_full)

# OOF map (leakage-safer export for modeling)
location_encoded_oof_df = build_oof_location_encoded(
    location_parts_df,
    y_loc_full,
    n_splits=5,
    alpha=35.0,
    random_state=42,
 )

for enc_df in (location_encoded_df, location_encoded_oof_df):
    if 'num__log_area_m2_v3' in model_base_df.columns:
        enc_df['num__locmean_x_logarea_v6'] = enc_df['num__location_te_mean_v6'] * model_base_df['num__log_area_m2_v3']
    if 'num__rooms' in model_base_df.columns:
        enc_df['num__locmean_x_rooms_v6'] = enc_df['num__location_te_mean_v6'] * model_base_df['num__rooms']

model_base_path = '../data/satilir_model_base_v3.csv'
location_parts_path = '../data/satilir_location_parts.csv'
location_encoded_path = '../data/satilir_location_encoded_v6.csv'
location_encoded_oof_path = '../data/satilir_location_encoded_oof_v6.csv'

model_base_df.to_csv(model_base_path, index=False)
location_parts_df.to_csv(location_parts_path, index=False)
location_encoded_df.to_csv(location_encoded_path, index=False)
location_encoded_oof_df.to_csv(location_encoded_oof_path, index=False)

print('Saved model base features:', model_base_path, '| shape:', model_base_df.shape)
print('Saved raw location parts:', location_parts_path, '| shape:', location_parts_df.shape)
print('Saved encoded location features (full-fit):', location_encoded_path, '| shape:', location_encoded_df.shape)
print('Saved encoded location features (OOF):', location_encoded_oof_path, '| shape:', location_encoded_oof_df.shape)
location_encoded_oof_df.head()

Saved model base features: ../data/satilir_model_base_v3.csv | shape: (4416, 51)
Saved raw location parts: ../data/satilir_location_parts.csv | shape: (4416, 5)
Saved encoded location features (full-fit): ../data/satilir_location_encoded_v6.csv | shape: (4416, 10)
Saved encoded location features (OOF): ../data/satilir_location_encoded_oof_v6.csv | shape: (4416, 10)


,num__city_te_v6,num__district_te_v6,num__neighborhood_te_v6,num__metro_te_v6,num__metro_present_v6,num__location_te_mean_v6,num__district_to_city_ratio_v6,num__metro_to_district_ratio_v6,num__locmean_x_logarea_v6,num__locmean_x_rooms_v6
0,171730.550344,238952.016561,179330.220742,258853.230543,1.0,212216.504547,1.391436,1.083285,889113.884614,636649.513642
1,172053.254339,245507.264770,179896.797849,235281.628484,1.0,208184.736360,1.426926,0.958349,834265.602676,416369.472721
2,173673.430802,170711.235372,144781.721788,112477.509204,0.0,150410.974292,0.982944,0.658876,694164.773411,601643.897166
3,107130.470176,105749.561378,179545.887369,112477.509204,0.0,126225.857032,0.987110,1.063622,480499.235201,126225.857032
4,108615.386787,106696.623419,179429.629097,112868.037740,0.0,126902.419261,0.982334,1.057841,539144.324434,380707.257782
